# 📌 Personally Identifiable Information (PII) Masking

![Topic](https://img.shields.io/badge/Topic-Masking-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Guardrails-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-May%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — **PII masking** is the practice of **hiding, replacing, or transforming personally identifiable information (names, emails, ID numbers, etc.)** so that sensitive data cannot be read or misused, while the data itself remains structurally useful. It is a **cornerstone of privacy regulations like GDPR and HIPAA.**</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install presidio_analyzer presidio_anonymizer` |


---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

## What is PII?
**PII** (Personally Identifiable Information) represent **any data that can be used to identify a specific individual**. For examples, full names, email addresses, social security numbers, phone numbers, home addresses, and medical records are PII data.
## What is masking?
**Data masking protects sensitive data by irreversibly replacing original, real data values with fictitious but realistic equivalents.** Masked data remains as useful for developers and testers as the original data values. 

Different tools are availble to mask your PII,such as Google Cloud DLP, AWS Comprehend, Microsoft Presidio, Azure Purview.

## Masking techniques
* **Redaction**: fully remove/replace with [REDACTED]
* **Pseudonymization**: Replace with a fake but consistent value (e.g. "John Smith" → "User_4821")
* **Tokenization**: Replace with a random token, keep a secure mapping elsewhere
* **Generalization**: Reduce precision (exact age → age range "30–40")
* **Hashing**: One-way transformation (irreversible, good for IDs)
* **Encryption**: Reversible transformation (authorized parties can decrypt)
* **Synthetic data replacement**: Replace data algorithmically generated information.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

1. Raw data enters the system (a database row, a text document, a form).
2. A PII detector scans it using pattern matching (regex for emails, phone numbers), Named Entity Recognition (NER for names, locations), and rule-based classifiers.
3. Each entity is classified by type (PERSON, EMAIL, SSN…) and risk level.
4. A masking technique is applied per entity type (see below).
5. Validation ensures the output is structurally correct (joins still work, formats are preserved).
6. Safe, masked output is returned for use in analytics, testing, or AI pipelines.

![Masking.png](../assets/masking.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Regulatory compliance** | Masking is the go-to mechanism for satisfying GDPR, HIPAA, and PCI-DSS requirements |
| 🟢 | **Data utility preserved** | Unlike full deletion, masked data retains its structure and statistical properties, making it suitable for testing, analytics, and AI training. |
| 🔴 | **Detection gaps** | Because Presidio and similar tools use automated detection mechanisms, there is no guarantee they will find all sensitive information. Additional systems and protections should be employed |
| 🔴 | **Re-identification risk** | Even masked data can sometimes be re-identified by combining multiple fields (e.g., a rare combination of age + job title + postcode), especially in small datasets |

---
## 4. Code Example

> **Goal:** Small examplre of PII masking usign Predisio library

In [3]:
# ---------------------------------------------------------------------------
# 1. Initialize the engines
# ---------------------------------------------------------------------------

# pip install presidio-analyzer presidio-anonymizer
# python -m spacy download en_core_web_lg

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# ---------------------------------------------------------------------------
# 1. Initialize the engines
# ---------------------------------------------------------------------------
analyzer  = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# ---------------------------------------------------------------------------
# 2. Sample text containing PII
# ---------------------------------------------------------------------------

text = "Hello, my name is John Smith. My email is john.smith@example.com."

# ---------------------------------------------------------------------------
# 3. Detect PII entities (returns a list of RecognizerResult objects)
# ---------------------------------------------------------------------------

results = analyzer.analyze(
    text=text,
    language="en",
    entities=["PERSON", "EMAIL_ADDRESS"]
)

print("Detected PII:")
for r in results:
    print(f"  [{r.entity_type}] '{text[r.start:r.end]}' (score: {r.score:.2f})") #Score represent the level of confidence of the model in its detection

# ---------------------------------------------------------------------------
# 4. Apply different masking strategies per entity type
# ---------------------------------------------------------------------------

operators = {
    "PERSON":       OperatorConfig("replace", {"new_value": "[REDACTED NAME]"}),
    "EMAIL_ADDRESS":OperatorConfig("mask",    {"masking_char": "*", "chars_to_mask": 20, "from_end": False})
}

anonymized = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators=operators
)

print("\nMasked output:")
print(anonymized.text)

# Expected output:
# Hello, my name is [REDACTED NAME]. My email is ********************com and my phone is .

Detected PII:
  [EMAIL_ADDRESS] 'john.smith@example.com' (score: 1.00)
  [PERSON] 'John Smith' (score: 0.85)

Masked output:
Hello, my name is [REDACTED NAME]. My email is ********************om.


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **PII masking transforms sensitive data into safe equivalents without destroying its structure or utility.**
- **Multiple techniques exist. Each suited to different risk levels and use cases.**
- **Automated detection is imperfect. No tool guarantees 100% recall, so human checking is always necessary.**
- **Masking is a legal necessity, not just a best practice**

</div>